# Algoritmos de Regresión

In [2]:
from pyspark.sql import SparkSession

print("Inicializando entorno y cargando datos etiquetados de la Semana 10...")

# 1. Iniciamos Spark en este nuevo notebook
spark = SparkSession.builder.appName("Semana12_Supervisados").getOrCreate()

# 2. Leemos DIRECTAMENTE los datos en formato Parquet local
ruta_datos = "/home/jovyan/work/semanas/Semana 10/modelos/datos_etiquetados_kmeans"
df_clusters = spark.read.parquet(ruta_datos)

# 3. Comprobamos la carga usando los nombres reales del esquema
print("\n--- Primeros 10 registros cargados exitosamente ---")
df_clusters.select("marca", "precio_kg", "rating", "opiniones", "prediction").show(10)

Inicializando entorno y cargando datos etiquetados de la Semana 10...

--- Primeros 10 registros cargados exitosamente ---
+-----+-----------------+-----------------+---------+----------+
|marca|        precio_kg|           rating|opiniones|prediction|
+-----+-----------------+-----------------+---------+----------+
|    0|9.020000457763672|4.800000190734863|      414|         1|
|    1|6.860000133514404|4.699999809265137|      100|         0|
|    3|5.150000095367432|4.699999809265137|      241|         0|
|    1|5.349999904632568|4.699999809265137|      155|         0|
|    0|7.639999866485596|4.800000190734863|      341|         1|
|    1|9.359999656677246|4.900000095367432|       21|         0|
|    3|4.670000076293945|4.800000190734863|      129|         0|
|    0|7.510000228881836|4.800000190734863|      673|         1|
|    3|4.670000076293945|4.699999809265137|      134|         0|
|    0|8.739999771118164|4.800000190734863|      444|         1|
+-----+-----------------+-------

In [3]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

print("Preparando vectores escalados y entrenando Regresión Lineal...")

# 1. Creamos el VectorAssembler (sin los precios)
assembler_regresion = VectorAssembler(
    inputCols=["rating", "opiniones"], 
    outputCol="features_regresion"
)
df_vector_reg = assembler_regresion.transform(df_clusters)

# 2. Escalamos las características para normalizar las magnitudes
scaler_reg = StandardScaler(inputCol="features_regresion", outputCol="scaledFeatures_regresion")
scaler_model_reg = scaler_reg.fit(df_vector_reg)
df_para_regresion = scaler_model_reg.transform(df_vector_reg)

# 3. Renombramos la variable objetivo Y
df_para_regresion = df_para_regresion.withColumnRenamed("precio_kg", "label_precio")

# =======================================================================
# Borramos la columna prediction del K-Means 
# para que no choque con la columna de salida de la Regresión Lineal
df_para_regresion = df_para_regresion.drop("prediction")
# =======================================================================

# 4. Dividimos en Entrenamiento (70%) y Prueba (30%) de forma reproducible
train_reg, test_reg = df_para_regresion.randomSplit([0.7, 0.3], seed=42)

# 5. Instanciamos y entrenamos el modelo con las características escaladas
lr_scaled = LinearRegression(featuresCol="scaledFeatures_regresion", labelCol="label_precio")
lr_model_scaled = lr_scaled.fit(train_reg)

# 6. Evaluamos en el set de prueba
predictions_final_reg = lr_model_scaled.transform(test_reg)

evaluator_r2 = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="r2")
evaluator_rmse = RegressionEvaluator(labelCol="label_precio", predictionCol="prediction", metricName="rmse")

r2_scaled = evaluator_r2.evaluate(predictions_final_reg)
rmse_scaled = evaluator_rmse.evaluate(predictions_final_reg)

# 7. Desplegamos los resultados oficiales de la Semana 12
print("\n==================================================")
print("  MÉTRICAS OFICIALES DE REGRESIÓN (DATOS ESCALADOS) ")
print("==================================================")
print(f"R² (Coeficiente de determinación): {r2_scaled:.4f}")
print(f"RMSE (Error promedio en precio):   ${rmse_scaled:.4f}")
print("==================================================")

Preparando vectores escalados y entrenando Regresión Lineal...

  MÉTRICAS OFICIALES DE REGRESIÓN (DATOS ESCALADOS) 
R² (Coeficiente de determinación): 0.0568
RMSE (Error promedio en precio):   $2.8363


In [5]:
from pyspark.ml.regression import LinearRegression

print("Configurando y entrenando el modelo definitivo de Regresión Lineal...")

# 1. Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# 2. Entrenar el modelo con los datos de entrenamiento (70%)
lr_reg_model = lr_regresion.fit(train_reg)

# 3. Hacer las predicciones sobre los datos de prueba (30%)
predictions_regresion = lr_model_scaled.transform(test_reg)

# 4. Mostrar las predicciones junto al precio real
print("\n=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(20)

Configurando y entrenando el modelo definitivo de Regresión Lineal...

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0| 8.739999771118164|7.410870973686384|
|    0| 8.739999771118164|7.410870973686384|
|    0| 8.739999771118164|7.410870973686384|
|    0| 8.739999771118

In [8]:
from pyspark.ml.regression import LinearRegression

print("Configurando y entrenando el modelo definitivo de Regresión Lineal...")

# 1. Configurar el modelo de Regresión Lineal
lr_regresion = LinearRegression(
    featuresCol="scaledFeatures_regresion", 
    labelCol="label_precio", 
    maxIter=10
)

# 2. Entrenar el modelo con los datos de entrenamiento (70%)
lr_reg_model = lr_regresion.fit(train_reg)

# 3. Corregido: Hacer las predicciones usando el modelo actual de la celda
predictions_regresion = lr_reg_model.transform(test_reg)

# 4. Mostrar las predicciones junto al precio real
print("\n=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===")
predictions_regresion.select("marca", "label_precio", "prediction").show(10)

Configurando y entrenando el modelo definitivo de Regresión Lineal...

=== COMPARATIVA: PRECIO REAL VS PRECIO PREDICHO ===
+-----+------------------+-----------------+
|marca|      label_precio|       prediction|
+-----+------------------+-----------------+
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|6.887154858970275|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
|    0|12.569999694824219|7.317322158572727|
+-----+------------------+-----------------+
only showing top 10 rows



In [9]:
# Imprimir la intersección (b) y los coeficientes (m) de la recta
print("=== COEFICIENTES DE LA ECUACIÓN DE REGRESIÓN ===")
print(f"Intersección (Precio base b): {lr_reg_model.intercept:.4f}")
print(f"Coeficiente de 'rating' (m1):    {lr_reg_model.coefficients[0]:.4f}")
print(f"Coeficiente de 'opiniones' (m2): {lr_reg_model.coefficients[1]:.4f}")

=== COEFICIENTES DE LA ECUACIÓN DE REGRESIÓN ===
Intersección (Precio base b): -14.6993
Coeficiente de 'rating' (m1):    0.6410
Coeficiente de 'opiniones' (m2): 0.0421
